## creating sentence embeddings for documentation from excel file

Convert sentences from an Excel file into embeddings using the saved model from Hugging Face (which we fine-tuned in the fine_tune_hugging_face_model.ipynb notebook).

In [ ]:
import sentence_transformers as st
import pandas as pd
import os

In [ ]:
chunk_size = 5
documents = pd.read_excel(os.path.join(os.path.dirname(os.getcwd()), 'data', 'documentation.xlsx'), index_col = 0)
model = st.SentenceTransformer('model_2')

# loading notes with assigned vectors representing a meaning
documentation_embeddings_path = os.path.join(os.path.dirname(os.getcwd()), 'data', 'documentation_embeddings.json')
if not os.path.isfile(documentation_embeddings_path):
    doc_embed = pd.DataFrame(columns = ['note', 'embeddings'], data = [])
else:
    doc_embed = pd.read_json(documentation_embeddings_path)

In [ ]:
# dividing text into chunks
def divide_text(text, chunk_size):
    # chunk_size is a number of words per chunk
    chunks = []
    for i in range(
        0, 
        len(text.split()) - (len(text.split()) % chunk_size), 
        chunk_size // 2
    ):
        chunks.append(' '.join(text.split()[i : i + chunk_size]))

    chunks.append(' '.join(text.split()[len(text.split()) - chunk_size : ]))
    
    return chunks

In [ ]:
# creating sentence embedding for given text. If this text is long the it is divided into smaller chunks
# and it creates meaning vector for each chunk
def encode_text(text, model):
    # if there is a lot of text then divide it into smaller chunks
    if len(text) > chunk_size:
        chunks = divide_text(text, chunk_size)
        embeddings = [model.encode(chunk) for chunk in chunks]
    else:
        embeddings = [model.encode(text)]
        
    return embeddings

In [ ]:
# take only those rows from documents which are not already in doc_embed
new_notes = documents.merge(doc_embed, on = 'note', how = 'left')
new_notes = new_notes[new_notes.embeddings.isnull()][['note']]
new_notes['embeddings'] = new_notes.note.apply(lambda x: encode_text(x, model))

In [ ]:
new_notes

,note,meaning_vectors
0,table Stage.dbo.vw_CCH_V6a contains informatio...,"[[0.12364568, -0.61376524, -0.20345598, 0.1331..."
1,table Stage.emp.tbl_TSProductivityEntries cont...,"[[-0.21699308, -0.38194418, -0.40946165, 0.275..."
2,table Stage.emp.tbl_Users contains information...,"[[0.13072844, -0.002587445, -0.2589916, 0.1895..."
3,table Stage.emp.tblLU_TSProductivityMeasures c...,"[[-0.27291617, -0.31101954, -0.33841905, 0.055..."
4,table Stage.emp.tbl_TSProductivityTimeSpans co...,"[[-0.20024124, -0.029982992, -0.313974, 0.2632..."
5,table Stage.emp.tbl_TSTimesheet contains infor...,"[[-0.085649654, -0.17372683, -0.32084945, 0.19..."
6,table Stage.emp.tblLU_TSOperations contains in...,"[[-0.25846297, -0.691412, -0.22136524, -0.0092..."
7,table Stage.emp.tblLU_TSOperationTypes contain...,"[[-0.2881527, -0.774925, -0.21019462, 0.013173..."
8,table Stage.emp.tbl_Clients contains informati...,"[[0.24958713, -0.23892462, -0.34755993, 0.1161..."
9,table Stage.emp.tblLU_Departments contains inf...,"[[0.1777846, -0.041062236, -0.28553337, 0.1533..."


In [ ]:
doc_embed = pd.concat((doc_embed, new_notes))
doc_embed.reset_index(drop = True, inplace = True)

In [ ]:
doc_embed.shape

(158, 2)

In [ ]:
doc_embed.to_json(documentation_embeddings_path)